# Examen: perceptrón de una capa, regresión lineal y regresión logística

**Materia:** Matemáticas Aplicadas a Ciencia de Datos

En este examen implementarás el mismo flujo visto en clase, pero usando el dataset **Diabetes** incluido en `sklearn.datasets`.

El objetivo no es estudiar validación ni generalización. El objetivo es mostrar que una estructura neuronal simple puede representar modelos estadísticos tradicionales:

- Regresión lineal.
- Regresión logística binaria.
- Perceptrón de una sola capa entrenado con descenso del gradiente.

## Instrucciones generales

Completa las celdas marcadas con `TODO`.

Puedes consultar el notebook de clase, pero debes adaptar el flujo al dataset Diabetes.

En este examen puedes entrenar y evaluar con los mismos datos. Esto se permite porque el foco está en la comparación matemática entre modelos, no en la evaluación predictiva fuera de muestra.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from src.lib import (
    nn_model_regression,
    nn_model_classification,
    forward_propagation_regression,
    forward_propagation_classification,
)

## 1. Carga de datos

Carga el dataset Diabetes desde `sklearn.datasets`.

Indicaciones:

- Importa la función `load_diabetes` desde `sklearn.datasets`.
- Carga los datos usando el argumento `as_frame=True`.
- Guarda el DataFrame completo en una variable llamada `df_diabetes`.
- Revisa las primeras filas con `.head()`.

El DataFrame debe contener las variables de entrada y una columna objetivo llamada `target`.

In [4]:
# TODO 1: carga el dataset Diabetes y crea df_diabetes
# Debe existir una columna llamada "target".

from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
df_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df_diabetes['target'] = diabetes.target

df_diabetes.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 2. Crear una variable de clasificación

El dataset Diabetes tiene un `target` numérico. Lo usaremos de dos formas:

- Como variable numérica para regresión.
- Como base para crear una variable binaria de clasificación.

Crea una columna llamada `target_cat` con la siguiente regla:

$$
target\_cat =
\begin{cases}
1 & \text{si } target > \text{mediana}(target) \\
0 & \text{si } target \leq \text{mediana}(target)
\end{cases}
$$

In [6]:
# TODO 2: calcula la mediana de target y crea la columna target_cat


target_threshold = df_diabetes['target'].median()
df_diabetes["target_cat"] = (df_diabetes["target"] > target_threshold).astype(int)

print(f'median {target_threshold}')

df_diabetes[["target", "target_cat"]].head()

median 140.5


,target,target_cat
0,151.0,1
1,75.0,0
2,141.0,1
3,206.0,1
4,135.0,0


**Pregunta 1.** ¿Por qué el mismo dataset permite formular un problema de regresión y uno de clasificación?


porque al definir un umbral sobre la variable target que en principio puede ser usada para regresión para decir que algo es o no es, se convierte en un problema binario que puede ser analizado por un modelo de clasificacion, buscando la probabilidad de que pertenezca o no a uuna clase.

## 3. Separar variables y estandarizar

Define:

- `X`: matriz de variables de entrada.
- `y_num`: target numérico.
- `y_cat`: target binario.

Después, estandariza `X` con `StandardScaler`.

In [7]:
# TODO 3: separa X, y_num y y_cat
# X no debe incluir ni "target" ni "target_cat".

X = df_diabetes.drop(columns=['target', 'target_cat'])
y_num = df_diabetes['target']
y_cat = df_diabetes['target_cat']

# TODO 4: estandariza X

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print("X_std shape:", X_std.shape)

X_std shape: (442, 10)


## 4. Convertir a forma matricial

El código del perceptrón espera:

$$X \in \mathbb{R}^{n_x \times m}$$

Por eso debes transponer la matriz de entrada.

También debes convertir los targets a forma:

$$Y \in \mathbb{R}^{1 \times m}$$

In [8]:
# TODO 5: convierte X_std, y_num y y_cat al formato requerido por src.lib

X_array = X_std.T
y_num_array = y_num.values.reshape(1, -1)
y_cat_array = y_cat.values.reshape(1,-1)

print("X_array shape:", X_array.shape)
print("y_num_array shape:", y_num_array.shape)
print("y_cat_array shape:", y_cat_array.shape)

X_array shape: (10, 442)
y_num_array shape: (1, 442)
y_cat_array shape: (1, 442)


**Pregunta 2.** Si el dataset tiene 10 variables de entrada y 442 observaciones, ¿cuáles deben ser las dimensiones de `X_array`, `W`, `b` y `Y_hat`?

tenemos en principio 10 variables y 442 observaciones, quiere decir que los pesos deben de ser de 1x10 dimensiones, asumiendo que el perceptrion es de una capa en este momento.

X_array es de 10x442 porque en este caso, cada fila representa una variable y cada columna una observacion.

El sesgo, como solo es un perceptron, solo es 1, entonces es 1x1.

las predicciones deben ser de 1x442, es decir, 442 predicciones ya sin las variables.

## 5. Perceptrón para regresión

Para regresión, el modelo usa salida identidad:

$$\hat{Y} = WX + b$$

La función de costo es MSE en forma escalada:

$$J(W,b) = \frac{1}{2m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2$$

In [10]:
# TODO 6: entrena el perceptrón de regresión

parameters_regression = nn_model_regression(
    X_array, 
    y_num_array, 
    num_iterations=500, #dado el numero de datos, pero igual se pueden hacer mas pruebas
    learning_rate=0.1, 
    print_cost=True
)

W_regression = parameters_regression["W"]
b_regression = parameters_regression["b"]

print("W:", W_regression)
print("b:", b_regression)

Costo después de la iteración 0: 14537.780026
Costo después de la iteración 1: 11628.811355
Costo después de la iteración 2: 9541.376036
Costo después de la iteración 3: 7947.349431
Costo después de la iteración 4: 6691.287973
Costo después de la iteración 5: 5686.761240
Costo después de la iteración 6: 4877.914002
Costo después de la iteración 7: 4224.597411
Costo después de la iteración 8: 3696.138249
Costo después de la iteración 9: 3268.371006
Costo después de la iteración 10: 2921.978591
Costo después de la iteración 11: 2641.415895
Costo después de la iteración 12: 2414.134654
Costo después de la iteración 13: 2229.991005
Costo después de la iteración 14: 2080.778882
Costo después de la iteración 15: 1959.857261
Costo después de la iteración 16: 1861.850402
Costo después de la iteración 17: 1782.406054
Costo después de la iteración 18: 1718.000132
Costo después de la iteración 19: 1665.778800
Costo después de la iteración 20: 1623.430764
Costo después de la iteración 21: 1589.083

In [11]:
print(y_num.mean())

152.13348416289594


In [13]:
# TODO 7: genera predicciones y calcula métricas de regresión

y_pred_regression = forward_propagation_regression(X_array, parameters_regression)

r2 = r2_score(y_num_array.ravel(), y_pred_regression.ravel())
mse = mean_squared_error(y_num_array.ravel(), y_pred_regression.ravel())
rmse = root_mean_squared_error(y_num_array.ravel(), y_pred_regression.ravel())
mae = mean_absolute_error(y_num_array.ravel(), y_pred_regression.ravel())

print(f"r2 score: {r2}")
print(f"mse: {mse}")
print(f"rmse: {rmse}")
print(f"mae: {mae}")

r2 score: 0.516144790312608
mse: 2869.205700216672
rmse: 53.56496709806394
mae: 43.33596468102469


**Pregunta 3.** Explica por qué este perceptrón de una capa es equivalente a una regresión lineal múltiple.


el perceptron en pocs palabras funciona multiplicando la entrada por un peso y sumando un sesgo, lo cual es muy parecido a la ecuacion de una regresion lineal multiple. Ahroa, lo que hace que actue como una regresion lineal multiple, es la funcion de activacion, ya que estas cuando solo es una capa del perceptron, son no lineales, combinando las entradas linealmente.

## 6. Comparación con `LinearRegression`

Entrena un modelo `LinearRegression` usando los mismos datos estandarizados y compara sus métricas con las del perceptrón.

In [14]:
# TODO 8: entrena LinearRegression y calcula r2

model_regression = LinearRegression()
model_regression.fit(X_std, y_num)

y_pred_regression_sklearn = model_regression.predict(X_std)
r2_sklearn = r2_score(y_num, y_pred_regression_sklearn)

print("Betas:", model_regression.coef_)
print("Beta0:", model_regression.intercept_)
print(f"r2 score sklearn: {r2_sklearn}")

Betas: [ -0.47612079 -11.40686692  24.72654886  15.42940413 -37.67995261
  22.67616277   4.80613814   8.42203936  35.73444577   3.21667372]
Beta0: 152.13348416289594
r2 score sklearn: 0.5177484222203498


**Pregunta 4.** ¿Por qué las métricas del perceptrón y de `LinearRegression` deberían ser parecidas, aunque no necesariamente idénticas?


porque de forma generalizada estan resolviendo el mimso problea y de la misma forma, aunque con ciertos matices.

ambos buscan minimizar la funcion de costo mse.

La razon por la qu eno son exactamente iguales son las formas de optimizacion que usen las librerias y el metodo en general, por ejemplo el descenso del gradiente o metodos analiticos e iterativos.

## 7. Perceptrón para clasificación binaria

Para clasificación, usamos la misma parte lineal:

$$Z = WX + b$$

pero ahora aplicamos una sigmoide:

$$A = \sigma(Z)$$

`A` puede interpretarse como una probabilidad.

In [17]:
# TODO 9: entrena el perceptrón de clasificación

parameters_classification = nn_model_classification(
    X_array, 
    y_cat_array, 
    num_iterations=500,
    learning_rate=0.4,
    print_cost=True
)

W_classification = parameters_classification["W"]
b_classification = parameters_classification["b"]

print("W:", W_classification)
print("b:", b_classification)

Costo después de la iteración 0: 0.697714
Costo después de la iteración 1: 0.613419
Costo después de la iteración 2: 0.575515
Costo después de la iteración 3: 0.554728
Costo después de la iteración 4: 0.541343
Costo después de la iteración 5: 0.531742
Costo después de la iteración 6: 0.524356
Costo después de la iteración 7: 0.518413
Costo después de la iteración 8: 0.513490
Costo después de la iteración 9: 0.509332
Costo después de la iteración 10: 0.505773
Costo después de la iteración 11: 0.502696
Costo después de la iteración 12: 0.500017
Costo después de la iteración 13: 0.497670
Costo después de la iteración 14: 0.495604
Costo después de la iteración 15: 0.493778
Costo después de la iteración 16: 0.492158
Costo después de la iteración 17: 0.490716
Costo después de la iteración 18: 0.489429
Costo después de la iteración 19: 0.488277
Costo después de la iteración 20: 0.487242
Costo después de la iteración 21: 0.486311
Costo después de la iteración 22: 0.485472
Costo después de la i

In [18]:
# TODO 10: genera probabilidades, conviértelas a clases y calcula métricas

prob_pred_classification = forward_propagation_classification(X_array, parameters_classification)
threshold = 0.5
y_pred_classification = (prob_pred_classification > threshold).astype(int)

accuracy = accuracy_score(y_cat_array.ravel(), y_pred_classification.ravel())
precision = precision_score(y_cat_array.ravel(), y_pred_classification.ravel())
recall = recall_score(y_cat_array.ravel(), y_pred_classification.ravel())
f1 = f1_score(y_cat_array.ravel(), y_pred_classification.ravel())

print(f"accuracy score: {accuracy}")
print(f"precision score: {precision}")
print(f"recall score: {recall}")
print(f"f1 score: {f1}")

accuracy score: 0.7420814479638009
precision score: 0.7399103139013453
recall score: 0.746606334841629
f1 score: 0.7432432432432432


**Pregunta 5.** ¿Por qué agregar una función sigmoide permite interpretar la salida como una probabilidad?

porque la funcion sigmoide toma cualquier valor numerico que salga de la combinacion lineal y lo aplasta o mapea estrictamente a un rango entre 0 y 1. como matematicamente los resultados siempre quedan atrapados en ese intervalo cumplen con la condicion fundamental de una probabilidad, donde 0 significa nula certeza y 1 significa certeza absoluta de que el paciente pertenece a la clase alta.

**Pregunta 6.** ¿Por qué necesitamos un umbral como `0.5` para obtener clases finales?

porque la red neuronal solo nos da una probabilidad continua y no una etiqueta directa.

el umbral funciona como una frontera de decision para desempatar: si la probabilidad es mayor a 0.5 el modelo decide clasificar al paciente como un 1, y si es menor o igual lo clasifica como un 0, transformando el valor continuo en una categoria binaria limpia.

## 8. Comparación con `LogisticRegression`

Entrena un modelo `LogisticRegression` usando los mismos datos estandarizados y compara sus métricas con las del perceptrón de clasificación.

In [19]:
# TODO 11: entrena LogisticRegression y calcula métricas

model_classification = LogisticRegression()
model_classification.fit(X_std, y_cat)
y_pred_classification_sklearn = model_classification.predict(X_std)

accuracy_sklearn = accuracy_score(y_cat, y_pred_classification_sklearn)
precision_sklearn = precision_score(y_cat, y_pred_classification_sklearn)
recall_sklearn = recall_score(y_cat, y_pred_classification_sklearn)
f1_sklearn = f1_score(y_cat, y_pred_classification_sklearn)

print("Betas:", ...)
print("Beta0:", ...)
print(f"accuracy score sklearn: {accuracy_sklearn}")
print(f"precision score sklearn: {precision_sklearn}")
print(f"recall score sklearn: {recall_sklearn}")
print(f"f1 score sklearn: {f1_sklearn}")

Betas: Ellipsis
Beta0: Ellipsis
accuracy score sklearn: 0.7420814479638009
precision score sklearn: 0.7399103139013453
recall score sklearn: 0.746606334841629
f1 score sklearn: 0.7432432432432432


**Pregunta 7.** Explica la diferencia entre usar salida identidad y usar salida sigmoide.

la salida identidad no le hace nada al resultado de la combinacion lineal por lo que te da un numero continuo que puede ser cualquiera desde menos infinito a mas infinito ideal para predecir valores exactos.

la salida sigmoide agarra ese mismo numero y lo aplasta para que a fuerza quede entre 0 y 1 transformandolo en una probabilidad que sirve para clasificar.

**Pregunta 8.** ¿Qué cambia entre regresión lineal y regresión logística desde el punto de vista de función de costo?

en la regresion lineal se usa el error cuadratico medio que mide la distancia al cuadrado entre el valor real y la prediccion buscando minimizar esa distancia.

en la regresion logistica se usa la entropia cruzada binaria que mide que tan lejos esta la probabilidad calculada de la clase real aplicando penalizaciones logaritmicas muy severas si el modelo se equivoca estando muy segu

**Pregunta 9.** ¿Qué papel tiene el descenso del gradiente en el perceptrón?

es el motor de aprendizaje del perceptron.

su papel es ir ajustando los pesos y el sesgo de forma iterativa calculando la derivada del error para saber hacia donde moverse y dar pasos hacia abajo en la curva del costo hasta encontrar los parametros optimos donde el error sea el minimo posible.

**Pregunta 10.** Explica con tus propias palabras por qué la regresión lineal y la regresión logística pueden verse como casos particulares de un perceptrón de una sola capa.

porque a arquitectura de la neurona es exactamente la misma en ambos modelos ya que los dos multiplican las variables por sus pesos y le suman el sesgo.

lo unico que cambia para que sea un caso o el otro es la activacion que decides poner al final: si dejas la salida tal cual sin transformar tienes la regresion lineal y si esa misma salida la pasas por la sigmoide tienes la regresion logistica.

